# Business Analyst Online Advertisement Prediction


Objective: Utilize SVM as a binary classifier to predict whether a user will clik on an online advertisement.

Importing the data file.

In [13]:
# We will import basic modules to start with and will import others as required as we go.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report

url = 'https://raw.githubusercontent.com/bobbhai69/ITEC203_Lab_repo/refs/heads/main/Lab4/Data/advertising.csv'
df = pd.read_csv(url)
df.head()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Ad Topic Line,City,Male,Country,Timestamp,Clicked on Ad
0,68.95,35,61833.90,256.09,Cloned 5thgeneration orchestration,Wrightburgh,0,Tunisia,2016-03-27 00:53:11,0
1,80.23,31,68441.85,193.77,Monitored national standardization,West Jodi,1,Nauru,2016-04-04 01:39:02,0
2,69.47,26,59785.94,236.50,Organic bottom-line service-desk,Davidton,0,San Marino,2016-03-13 20:35:42,0
3,74.15,29,54806.18,245.89,Triple-buffered reciprocal time-frame,West Terrifurt,1,Italy,2016-01-10 02:31:19,0
4,68.37,35,73889.99,225.58,Robust logistical utilization,South Manuel,0,Iceland,2016-06-03 03:36:18,0


## Task 1
### Data Cleaning
Removing 'Ad Topic Line' and 'Timestamp' from the dataframe.

In [14]:
# Removing columns 'Ad Topic Line' and 'Timestamp'

df_cleaned = df.drop(['Ad Topic Line', 'Timestamp'], axis=1)
df_cleaned.head()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,City,Male,Country,Clicked on Ad
0,68.95,35,61833.90,256.09,Wrightburgh,0,Tunisia,0
1,80.23,31,68441.85,193.77,West Jodi,1,Nauru,0
2,69.47,26,59785.94,236.50,Davidton,0,San Marino,0
3,74.15,29,54806.18,245.89,West Terrifurt,1,Italy,0
4,68.37,35,73889.99,225.58,South Manuel,0,Iceland,0


In [15]:
print(df_cleaned.columns)

Index(['Daily Time Spent on Site', 'Age', 'Area Income',
       'Daily Internet Usage', 'City', 'Male', 'Country', 'Clicked on Ad'],
      dtype='object')


### Data Preparation
Applying one-hot enocding to transform 'Country' and 'City' variables into numeric formats.

In [16]:
df_cleaned = pd.get_dummies(df_cleaned, columns=['Country', 'City'], drop_first=True)
for col in df_cleaned.select_dtypes(include='bool').columns:  # Convert boolean columns to integer
    df_cleaned[col] = df_cleaned[col].astype(int)
df_cleaned.head()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Male,Clicked on Ad,Country_Albania,Country_Algeria,Country_American Samoa,Country_Andorra,...,City_Wintersfort,City_Wongland,City_Wrightburgh,City_Wrightview,City_Yangside,City_Youngburgh,City_Youngfort,City_Yuton,City_Zacharystad,City_Zacharyton
0,68.95,35,61833.90,256.09,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,80.23,31,68441.85,193.77,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,69.47,26,59785.94,236.50,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,74.15,29,54806.18,245.89,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,68.37,35,73889.99,225.58,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Data Exploration
Summary Statistics

In [17]:
df_cleaned.describe()

,Daily Time Spent on Site,Age,Area Income,Daily Internet Usage,Male,Clicked on Ad,Country_Albania,Country_Algeria,Country_American Samoa,Country_Andorra,...,City_Wintersfort,City_Wongland,City_Wrightburgh,City_Wrightview,City_Yangside,City_Youngburgh,City_Youngfort,City_Yuton,City_Zacharystad,City_Zacharyton
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,65.000200,36.009000,55000.000080,180.000100,0.481000,0.50000,0.007000,0.006000,0.005000,0.002000,...,0.001000,0.001000,0.002000,0.001000,0.001000,0.001000,0.001000,0.001000,0.001000,0.001000
std,15.853615,8.785562,13414.634022,43.902339,0.499889,0.50025,0.083414,0.077266,0.070569,0.044699,...,0.031623,0.031623,0.044699,0.031623,0.031623,0.031623,0.031623,0.031623,0.031623,0.031623
min,32.600000,19.000000,13996.500000,104.780000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,51.360000,29.000000,47031.802500,138.830000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,68.215000,35.000000,57012.300000,183.130000,0.000000,0.50000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,78.547500,42.000000,65470.635000,218.792500,1.000000,1.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,91.430000,61.000000,79484.800000,269.960000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [18]:
# Check for missing values
df_cleaned.isnull().sum()

Daily Time Spent on Site    0
Age                         0
Area Income                 0
Daily Internet Usage        0
Male                        0
                           ..
City_Youngburgh             0
City_Youngfort              0
City_Yuton                  0
City_Zacharystad            0
City_Zacharyton             0
Length: 1210, dtype: int64

In [19]:
# Distribution of the target variable
df_cleaned['Clicked on Ad'].value_counts()

Clicked on Ad
0    500
1    500
Name: count, dtype: int64

## Task 2: Machine Learning with SVM

### Assigning Data
Designating 'Click on Ad' as the label (y) and the remaining variables as the features (X).

In [20]:
X = df_cleaned.drop('Clicked on Ad', axis=1)
y = df_cleaned['Clicked on Ad']

### Train Test Split
Splitting the dataset into training and testing sets.

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=83)

### Model Training and Prediction
Initializing and training the SVM classifier on the training data.

In [ ]:
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

### Model Evaluation

Using Confusion matrix and classification report, we evaluate our model's prediction.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)


Confusion Matrix:
[[143   2]
 [  8 147]]


In [ ]:
cr = classification_report(y_test, y_pred)
print("Classification Report:")
print(cr)

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       145
           1       0.99      0.95      0.97       155

    accuracy                           0.97       300
   macro avg       0.97      0.97      0.97       300
weighted avg       0.97      0.97      0.97       300



### Assessing the effectiveness of our SVM model

#### Interpretation of Confusion Matrix
The confusion matri is in the form:<br>
[[TN FP]<br>
[FN TP]]<br>
Which implies <br>
TN (True Negative)=143 <br>
FP (False Positive)=2 <br>
FN (False Negative)=8 <br>
TP (True Positive)=147 <br>

>This means only 10 out of 300 predictions were incorrect. The model correctly identified 143 users who did not click on the ad and 147 who did.

#### Interpretation of Classification Report
Precision: <br>
Class 0 (did not click on ad): 0.95<br>
Class 1 (clicked on ad): 0.99<br>
<br>
Recall: <br>
Class 0 (did not click on ad): 0.99<br>
Class 1 (clicked on ad): 0.95<br>
<br>
F1-Score:<br>
Both classes scored 0.97.<br>
The overall accuracy is 97%, which indicates strong generalization and minimal overfitting. High precision and recall for both classes suggest model is effective at distinguishing users who will either click on ads or not, which is crucial for targetted marketting.

The *Macro and Weighted averages* are 0.97 for all three metrics - precision, recall, and F1-score, which demonstrates the model's consistency. <br>To conclude, the SVM model is highly effective for this binary classification task with minimal missclassification and strong predictive accuracy.

Demographic and Geographic factors can have somewhere from moderate effect to high effect in predicting user engagement with online advertisements using a Support Vector Machine Mode. However the predictive power for these models is significantly dependent on the context and how these features can be engineered into the model.
<br>
It is more than obvious that demographic factors such as age, income, gender, education, etc and geographic factors such as location, region, etc have a direct relation with user interests, buying habits and online activities. These features can be used in the SVM model in targeted advertisements. SVM model with an appropriate kernel can perform well if demographic and geographic data reveal patterns that separate classes effectively in feature space.
<br>
However, that is not always the case. Because user engagement is also driven by behaviourial data which can vary within the same demographic and/or geographic groups.Using only demographic and geographical data may limit the model performance.
<br>
Therefore, combined use of demographic, geographic data with behaviourial and contextual data might boost the model performance and accuracy.
